Install dependencies and load data

In [14]:
!pip install transformers datasets scikit-learn torch accelerate tqdm -q


[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os, random, numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
from transformers import (
    RobertaTokenizerFast, RobertaForSequenceClassification,
)
import numpy as np

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_DIR   = '../training/roberta_classifier_final'
INFER_BATCH = 128
MAX_LEN      = 512
MODEL_NAME   = 'roberta-base'

LABEL2ID = {'not_green_claim': 0, 'green_claim': 1}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

c:\Users\willi\CPSC449\canada-oil-greenwash-scraping\greenwash-scraping\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
df = pd.read_csv("../../labelled/labelled.csv")
df = df[df["Label"] != -1]
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

unlabelled_df = pd.read_csv("../../labelled/unlabelled.csv")

# For duplicate sentences in unlabelled, give them the same label as the human labelling
label_map = dict(zip(df["Sentence"], df["Label"]))
unlabelled_df["Label"] = unlabelled_df["Sentence"].map(label_map).fillna(-1)

Label with pre-trained model

In [ ]:
tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME)

best_threshold = float(np.load(os.path.join(MODEL_DIR, "best_threshold.npy"))[0])

best_model = RobertaForSequenceClassification.from_pretrained(
    os.path.abspath(MODEL_DIR)
).to(DEVICE)
best_model.eval()

to_label = unlabelled_df["Label"] == -1
sentences = unlabelled_df.loc[to_label, "Sentence"].tolist()

all_binary_labels = []

for i in tqdm(range(0, len(sentences), INFER_BATCH), desc="Labelling"):
    batch_texts = sentences[i : i + INFER_BATCH]
    enc = tokenizer(
        batch_texts,
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    ).to(DEVICE)

    with torch.no_grad():
        probs = torch.softmax(best_model(**enc).logits, dim=-1)[:, 1].cpu().numpy()

    labels = (probs > best_threshold).astype(int)
    all_binary_labels.extend(labels.tolist())

unlabelled_df.loc[to_label, "Label"] = all_binary_labels

print(f"Total labelled: {len(sentences)}")
print(f"Green claims: {(sum(all_binary_labels))}")
print(f"Not green claims: {(len(all_binary_labels) - sum(all_binary_labels))}")

In [ ]:
combined_df = pd.concat([df, unlabelled_df], ignore_index=True)
combined_df = combined_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
combined_df.to_csv("../../output/analyzed/all_labelled.csv", index=False)

In [2]:
combined_df = pd.read_csv("../../output/analyzed/all_labelled.csv")

Basic dataset summary statistics

In [3]:
_PERIOD = {True: "pre-legislation", False: "post-legislation"}
_METRIC = {
    "green_claims": "Green Claims",
    "total_rows": "Total Sentences",
    "normalized": "Normalized",
}

_COL_ORDER = [f"{m} ({p})" for m in _METRIC.values() for p in _PERIOD.values()]


def summary_table(df):
    return df.agg(
        **{"Total Sentences": ("Label", "count"), "Green Claims": ("Label", "sum")}
    ).assign(Normalized=lambda x: (x["Green Claims"] / x["Total Sentences"]).round(4))


wayback_table = summary_table(combined_df.groupby("isWayback"))
print("=== Before vs. After Bill C-59 ===")
print(wayback_table.to_string())

org_table = summary_table(combined_df.groupby("Organization")).sort_values(
    "Green Claims", ascending=False
)
print("\n=== By Organization ===")
print(org_table.to_string())

org_wayback_pivot = (
    combined_df.groupby(["Organization", "isWayback"])
    .agg(green_claims=("Label", "sum"), total_rows=("Label", "count"))
    .assign(normalized=lambda x: (x["green_claims"] / x["total_rows"]).round(4))
    .unstack("isWayback")
)

org_wayback_pivot.columns = [
    f"{_METRIC[m]} ({_PERIOD[wb]})" for m, wb in org_wayback_pivot.columns
]
org_wayback_pivot = org_wayback_pivot[_COL_ORDER]

print("\n=== Green Claims by Organization (pre-legislation vs post-legislation) ===")
print(org_wayback_pivot.to_string())

=== Before vs. After Bill C-59 ===
           Total Sentences  Green Claims  Normalized
isWayback                                           
False                23127          1598      0.0691
True                 44749          4022      0.0899

=== By Organization ===
                            Total Sentences  Green Claims  Normalized
Organization                                                         
Enbridge                              12214          1638      0.1341
Suncor Energy                         15346          1339      0.0873
Pembina Pipeline                      25341          1101      0.0434
Shell Canada                           2275           559      0.2457
Canadian Natural Resources             7691           534      0.0694
Imperial Oil                           5009           449      0.0896

=== Green Claims by Organization (pre-legislation vs post-legislation) ===
                            Green Claims (pre-legislation)  Green Claims (post-legislation) 